# E3-B — Random Forest & XGBoost, multi-horizon (15/30/60/120 min)

**Extension indépendante** du protocole initial. Les modèles sont entraînés sur l’intégralité du train de chaque horizon, sélectionnés sur validation uniquement et évalués sur les **mêmes clés de test**. Les hyperparamètres candidats sont ceux du notebook historique `08A_PERSISTENCE_RIDGE_RF_XGBOOST(5)`. Les résultats originaux ne sont pas modifiés.

**Attention :** la validation de E3_09 doit avoir réussi. Exécuter toutes les cellules dans l’ordre. Environ 32 entraînements d'ensembles d'arbres (4 horizons × 8 candidats), potentiellement coûteux.

In [ ]:
from pathlib import Path
import json, time, platform
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
ROOT=Path.cwd().resolve()
REP=ROOT/'outputs/E3_aligned_representations'
COMMON=ROOT/'outputs/E3_common_evaluation'
OUT=ROOT/'outputs/E3_10B_RF_XGBOOST_multihorizon'
OUT.mkdir(parents=True,exist_ok=True)
HORIZONS=[15,30,60,120]; SPLITS=['train','validation','test']; KEYS=['uuid','timestamp','split']; SEED=42
for p in [REP,COMMON,ROOT/'outputs/E3_pretraining_audit']:
    assert p.exists(),f'Missing prerequisite: {p}'
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp']=pd.to_datetime(common['timestamp'],utc=True)
assert len(common)==91763 and not common.duplicated(KEYS).any()
assert common.groupby('split').size().to_dict()=={'train':64393,'validation':13727,'test':13643}
print('Output:',OUT,'| xgboost:',xgb.__version__)

In [ ]:
def load_split(h,s):
    folder=REP/f'h{h:03d}'
    m=pd.read_csv(folder/f'metadata_{s}.csv')
    m['timestamp']=pd.to_datetime(m['timestamp'],utc=True)
    m['target_timestamp']=pd.to_datetime(m['target_timestamp'],utc=True)
    assert m['split'].eq(s).all() and not m.duplicated(KEYS).any()
    assert (m['target_timestamp']-m['timestamp']).eq(pd.Timedelta(minutes=h)).all()
    x=np.load(folder/f'X_tabular_{s}_scaled.npy',mmap_mode='r')
    y=np.load(folder/f'y_tabular_{s}.npy',mmap_mode='r')
    xu=np.load(folder/f'X_tabular_{s}_unscaled.npy',mmap_mode='r')
    assert x.shape==(len(m),15) and len(y)==len(m)==len(xu)
    assert np.isfinite(x).all() and np.isfinite(y).all()
    assert np.allclose(m['target'],y,rtol=0,atol=1e-5)
    idx=np.load(COMMON/f'indices_h{h:03d}_{s}.npy')
    assert len(idx)==len(set(idx.tolist())) and (idx>=0).all() and (idx<len(m)).all()
    actual=m.iloc[idx][KEYS].sort_values(KEYS).reset_index(drop=True)
    expected=common.loc[common.split.eq(s),KEYS].sort_values(KEYS).reset_index(drop=True)
    assert actual.equals(expected),f'Key mismatch {h} {s}'
    return m,np.asarray(x),np.asarray(y),np.asarray(xu[:,12]),idx

def metrics(y,p):
    y=np.asarray(y);p=np.asarray(p)
    return {'n':len(y),'RMSE':float(np.sqrt(mean_squared_error(y,p))),
            'MAE':float(mean_absolute_error(y,p)), 'R2':float(r2_score(y,p)),
            'Bias':float(np.mean(p-y))}


In [ ]:
# Candidate grids copied from historical notebook 08A (5)
rf_candidates=[
    {'n_estimators':300,'max_depth':None,'min_samples_leaf':1,'max_features':1.0},
    {'n_estimators':300,'max_depth':20,'min_samples_leaf':2,'max_features':1.0},
    {'n_estimators':300,'max_depth':12,'min_samples_leaf':2,'max_features':1.0},
    {'n_estimators':300,'max_depth':20,'min_samples_leaf':4,'max_features':0.8},
]
xgb_candidates=[
    {'n_estimators':300,'max_depth':3,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
    {'n_estimators':500,'max_depth':3,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
    {'n_estimators':300,'max_depth':5,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
    {'n_estimators':500,'max_depth':5,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
]
def build_model(name,params):
    if name=='Random Forest':
        return RandomForestRegressor(**params,random_state=SEED,n_jobs=-1)
    return xgb.XGBRegressor(**params,random_state=SEED,n_jobs=-1,
                            objective='reg:squarederror',tree_method='hist',verbosity=0)


In [ ]:
search=[]; selected=[]; primary=[]; pred_frames=[]; device_rows=[]; split_rows=[]
for h in HORIZONS:
    mtr,xtr,ytr,_,_=load_split(h,'train')
    mva,xva,yva,_,_=load_split(h,'validation')
    mte,xte,yte,tlast,ite=load_split(h,'test')
    persistence=tlast[ite]
    baseline=metrics(yte[ite],persistence)['RMSE']
    for name,candidates in [('Random Forest',rf_candidates),('XGBoost',xgb_candidates)]:
        best=None
        for j,params in enumerate(candidates):
            start=time.perf_counter()
            model=build_model(name,params).fit(xtr,ytr)
            pv=model.predict(xva)
            score=metrics(yva,pv)['RMSE']
            elapsed=time.perf_counter()-start
            search.append({'horizon_min':h,'model':name,'candidate':j+1,
                           'params_json':json.dumps(params),'validation_RMSE':score,
                           'train_seconds':elapsed})
            print(f'h={h:3d} {name:13s} candidate={j+1} validation RMSE={score:.6f} ({elapsed:.1f}s)',flush=True)
            if best is None or score<best[0]:best=(score,j,params,model,elapsed)
        _,j,params,model,_=best
        # Refit on the full original TRAIN only; no validation/test leakage.
        start=time.perf_counter()
        final=build_model(name,params).fit(xtr,ytr)
        elapsed=time.perf_counter()-start
        selected.append({'horizon_min':h,'model':name,'candidate':j+1,
                         'params_json':json.dumps(params),'train_samples':len(ytr),
                         'validation_samples':len(yva),'training_seconds':elapsed})
        for s,x,y in [('train',xtr,ytr),('validation',xva,yva),('test',xte,yte)]:
            p=final.predict(x)
            split_rows.append({'horizon_min':h,'split':s,'model':name,**metrics(y,p)})
        pt=final.predict(xte[ite])
        result=metrics(yte[ite],pt)
        primary.append({'horizon_min':h,'model':name,**result,
                        'persistence_RMSE':baseline,'skill_RMSE':1-result['RMSE']/baseline,
                        'selected_candidate':j+1})
        f=mte.iloc[ite][['uuid','timestamp','target_timestamp']].copy().reset_index(drop=True)
        f['horizon_min']=h;f['model']=name;f['y_true']=yte[ite];f['y_pred']=pt
        f['persistence_pred']=persistence
        pred_frames.append(f)
        for uuid,g in f.groupby('uuid'):
            d=metrics(g.y_true,g.y_pred)
            ref=metrics(g.y_true,g.persistence_pred)['RMSE']
            device_rows.append({'horizon_min':h,'model':name,'uuid':uuid,**d,
                                'persistence_RMSE':ref,'skill_RMSE':1-d['RMSE']/ref if ref>0 else np.nan})
    print('Finished horizon',h,flush=True)


In [ ]:
pd.DataFrame(primary).sort_values(['horizon_min','model']).to_csv(OUT/'00_PRIMARY_common_TEST_results.csv',index=False)
pd.DataFrame(split_rows).to_csv(OUT/'01_metrics_all_splits.csv',index=False)
pd.DataFrame(search).to_csv(OUT/'02_validation_search.csv',index=False)
pd.concat(pred_frames,ignore_index=True).to_csv(OUT/'03_common_test_predictions.csv',index=False)
pd.DataFrame(device_rows).to_csv(OUT/'04_common_test_per_device.csv',index=False)
pd.DataFrame(selected).to_csv(OUT/'05_selected_configs_and_training.csv',index=False)
(OUT/'06_protocol.json').write_text(json.dumps({
    'source':'E3_aligned_representations','common_keys':'E3_common_evaluation',
    'horizons_minutes':HORIZONS,'seed':SEED,'feature_count':15,
    'training':'full horizon-specific train','model_selection':'validation RMSE only',
    'evaluation':'common test indices, 13643 rows per horizon',
    'historical_results':'unchanged','xgboost_version':xgb.__version__,
    'python':platform.python_version()},indent=2),encoding='utf-8')
print(pd.DataFrame(primary).sort_values(['horizon_min','model']).to_string(index=False))
print('Saved to',OUT)


### Après exécution
Transmettre `00_PRIMARY_common_TEST_results.csv`, `02_validation_search.csv`, `04_common_test_per_device.csv` et, si possible, `05_selected_configs_and_training.csv`.

**Limites :** les performances ne sont pas des preuves d’un effet causal de la représentation; les modèles à arbres peuvent être sensibles au décalage saisonnier. Une comparaison statistique et une analyse temporelle complémentaire restent nécessaires.